# TAPAS effective-epsilon audit for DPGAN — Colab runner

Runs `run_dpgan_effeps.py` on a GPU runtime. Before using this notebook:

1. Upload `common.py` and `dpgan/run_dpgan_effeps.py` to your Drive at
   `MyDrive/VRI/experimentation/evaluation/eval_tapas/eff_eps/` (mirroring
   the repo's own relative layout: `common.py` directly under `eff_eps/`,
   `run_dpgan_effeps.py` under `eff_eps/dpgan/`).
2. Make sure `data/adult_train.csv` and `data/adult_test.csv` already exist
   at `MyDrive/VRI/experimentation/data/` (same place `eval_synthcity_colab.ipynb`
   uses).
3. Runtime > Change runtime type > select a GPU (e.g. T4) before running.

Cache (`cache/dpgan/threat_model.pkl`, `cache/result_*.json`) and results
(`effeps_dpgan.csv`, `ROC_curve_dpgan.png`, etc.) write straight through to
Drive as the script runs — a disconnect mid-run loses nothing; just re-open
this notebook and re-run the cells, and the script resumes exactly where it
left off (see `common.py`'s caching docs).

## 1. Install dependencies + restart runtime

Pins match `requirements.txt` exactly. Deliberately does **not** touch
`torch`/`torchvision` — Colab's preinstalled versions are already a
CUDA-enabled build matched to its GPU, and letting pip's resolver reinstall
them here would very likely silently swap in a CPU-only wheel (the same
class of bug just fixed locally, but worse here since it wouldn't even
error — `torch.cuda.is_available()` would just quietly return `False`).

In [ ]:
# TAPAS's pyproject.toml declares `python = ">=3.9, <3.11"` (poetry-core
# build backend) -- Colab runs Python 3.12+, which poetry-core rejects
# outright when computing build metadata ("Getting requirements to build
# wheel did not run successfully"). Clone the exact pinned commit and patch
# that constraint before installing, rather than installing directly from
# git (which can't be patched first).
!rm -rf /content/tapas_src
!git clone -q https://github.com/alan-turing-institute/tapas.git /content/tapas_src
!cd /content/tapas_src && git checkout -q a7069d7e040828db0da174d1b003fa03a98e5453
!sed -i 's/python = ">=3.9, <3.11"/python = ">=3.9"/' /content/tapas_src/pyproject.toml
!grep '^python =' /content/tapas_src/pyproject.toml  # sanity-check the patch applied

# TAPAS's pyproject.toml also declares pandas = "^1.4.1" (i.e. wants pandas
# <2.0). Old pandas 1.x releases have no prebuilt wheel for Python 3.12, so
# pip tries to build one from source and fails the same way ("Getting
# requirements to build wheel") -- this is the SAME conflict the README
# already documents for local installs ("TAPAS downgrades pandas; re-pin
# after install"), just failing loudly here instead of silently downgrading.
# We already know from the local PrivBayes run that TAPAS's actual code
# works fine with pandas==2.3.3 -- so skip TAPAS's dependency resolution
# entirely with --no-deps, and explicitly install the one TAPAS-specific
# package that wouldn't otherwise be covered by Colab's defaults or our
# other installs below.
!pip install /content/tapas_src --no-deps -q
!pip install palettable==3.3.3 -q

!pip install synthcity==0.2.12 -q
!pip install opacus==1.4.1 -q
!pip install pandas==2.3.3 -q

import os
os.kill(os.getpid(), 9)  # force restart to clear numpy/torch binary conflicts

In [ ]:
import tapas
import synthcity
from synthcity.plugins import Plugins
print("imports OK -- tapas, synthcity, and synthcity.plugins.Plugins all loaded successfully.")

## 2. Verify GPU + package versions

Run this **before** the actual DPGAN script. If `CUDA available` shows
`False`, stop here and fix the runtime type first — otherwise the script
will run to completion on CPU without any error, and you won't find out
until it's taken hours.

In [ ]:
import torch
import torchvision

print('torch version:      ', torch.__version__)
print('torchvision version: ', torchvision.__version__)
print('CUDA available:      ', torch.cuda.is_available())
print()

if not torch.cuda.is_available():
    print('*** WARNING: no CUDA GPU detected. ***')
    print('Go to Runtime > Change runtime type > select a GPU (e.g. T4), then re-run this cell.')
    print("Synthcity's dpgan plugin does NOT auto-detect CUDA -- run_dpgan_effeps.py")
    print('only actually uses the GPU if torch.cuda.is_available() is True here.')
else:
    print('GPU detected -- run_dpgan_effeps.py will pick up device="cuda" automatically.')

print()
!nvidia-smi

## 3. Mount Drive + symlink the eff_eps folder

Symlinking the **whole** `eff_eps/` folder (not just `cache/`) means:
- `common.py` and `dpgan/run_dpgan_effeps.py` are read directly from where
  you uploaded them on Drive — no copy step needed, and any edits you make
  on Drive are picked up automatically next run.
- `common.py` resolves its own paths via `Path(__file__).resolve()`, which
  follows this symlink through to the real Drive path — so `cache/dpgan/`,
  the log file, and (via `REPO_ROOT`) `data/` and `results/` all resolve
  straight onto Drive automatically. No separate results/data symlinks are
  needed for this script.

In [ ]:
from google.colab import drive
import os

drive.mount('/content/drive')

DRIVE_BASE = '/content/drive/MyDrive/VRI/experimentation'
os.chdir('/content')

# Make sure the Drive-side folders exist (no-op if you've already uploaded
# common.py/run_dpgan_effeps.py there).
os.makedirs(f'{DRIVE_BASE}/evaluation/eval_tapas/eff_eps/dpgan', exist_ok=True)
os.makedirs(f'{DRIVE_BASE}/data', exist_ok=True)

os.makedirs('/content/evaluation/eval_tapas', exist_ok=True)
if not os.path.exists('/content/evaluation/eval_tapas/eff_eps'):
    os.symlink(f'{DRIVE_BASE}/evaluation/eval_tapas/eff_eps',
               '/content/evaluation/eval_tapas/eff_eps')

print('Drive mounted and symlinked.')
print()
print('eff_eps/ contents:', os.listdir('/content/evaluation/eval_tapas/eff_eps'))

# Sanity-check the two required script files and the data files are actually there.
required = [
    f'{DRIVE_BASE}/evaluation/eval_tapas/eff_eps/common.py',
    f'{DRIVE_BASE}/evaluation/eval_tapas/eff_eps/dpgan/run_dpgan_effeps.py',
    f'{DRIVE_BASE}/data/adult_train.csv',
    f'{DRIVE_BASE}/data/adult_test.csv',
]
missing = [p for p in required if not os.path.exists(p)]
if missing:
    print()
    print('*** MISSING FILES -- upload these to Drive before running the script: ***')
    for p in missing:
        print(' ', p)
else:
    print()
    print('All required files found.')

## 4. Run the DPGAN effective-epsilon audit

See the runtime note at the top of `run_dpgan_effeps.py` for current
settings (`n_iter=100` for shadow models, `NUM_TRAIN=20`/`NUM_TEST=30`).
Safe to re-run this cell after a disconnect — cached simulations and
completed attacks are skipped automatically.

In [ ]:
!python evaluation/eval_tapas/eff_eps/dpgan/run_dpgan_effeps.py

## 5. Sanity-check the results

No "copy back to Drive" step needed — results were already written straight
to Drive via the symlink in step 3. This just reads them back to confirm.

In [ ]:
import pandas as pd

results_path = f'{DRIVE_BASE}/results/tapas_results/eff_eps_results/dpgan/effeps_dpgan.csv'
print(pd.read_csv(results_path))